In [ ]:
import sys
sys.path.append('../Model_info')
import os
import Stdinput as orctm
import netCDF4 as nc
import numpy as np
import time

In [ ]:
INI=orctm.Orctm_cvcm('ORCTM_cvcm','initial fields for Temperature')
latp_grid = INI.model_info_p('latp_grid')
lonp_grid = INI.model_info_p('lonp_grid')
zzout = INI.model_info_p('zzout')
mask = INI.model_info_p('weto_p')
mask = mask.astype(int)

In [ ]:
cmems_file = 'CMEMS-For-CVCM_INI_file.nc'
ini_file = nc.Dataset(cmems_file,mode='r',format='netCDF4')
#print(ini_file.variables)
latitude = ini_file.variables['latitude'][:]
longitude = ini_file.variables['longitude'][:]
longitude_grid,latitude_grid = np.meshgrid(longitude,latitude)
DEP = ini_file.variables['depth'][:]
to = ini_file.variables['thetao'][:,:,:,:]
to = to.squeeze()
to = to.filled(fill_value = np.nan)
nc.Dataset.close(ini_file)

In [ ]:
start_time = time.time()
# Start the timer
keo = np.size(to,axis=0)
to_keo = np.zeros((INI.ie_g,INI.je_g,keo))
to_ini = np.zeros((INI.ie_g,INI.je_g,INI.ke))
Lat_grid = INI.rearrange_grid_to_orctm(latitude_grid)
Lon_grid = INI.rearrange_grid_to_orctm(longitude_grid)
for k in range(keo):
    print('Layer: ',DEP[k],' m')
    to_grid  = INI.rearrange_grid_to_orctm(to[k,:,:])
    land = np.zeros_like(to_grid)
    land[np.isnan(to_grid)]=1  # land 1 Sea: 0 !!!!
    if np.sum(land) != np.size(land):
        to_keo[:,:,k] = INI.Grid_bilinear_interpolation( \
            to_grid,land,Lat_grid,Lon_grid,latp_grid,lonp_grid,landvalue=np.nanmean(to_grid))

to_keo[to_keo==0]=np.nan # due to the landmask
to_ini = INI.vertical_interp2d(to_keo,DEP,zzout,mask)
to_ini[mask==0] = np.nan
# check obc
# minimum
to_ini = np.where(to_ini <= np.nanmin(to[:,:,:]),np.nanmin(to[:,:,:]),to_ini)
# maximum
to_ini = np.where(to_ini >= np.nanmax(to[:,:,:]),np.nanmax(to[:,:,:]),to_ini)
# Print elapsed time
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.6f} seconds")
to_ini[np.isnan(to_ini)==True]=0

In [ ]:
import struct

if1 = 0
if22 = 2
with open('INITEM', 'wb') as initem:
    for k in range(INI.ke):
        print('Layer: ',int(INI.ZZOUT[k]),' m')
        # Header for INITEM
        # >q 
        header = struct.pack('>qqqq', int(if1), int(if22), int(INI.ZZOUT[k]), int(INI.ie_g * INI.je_g))
        header_len = len(header)
        
        # >i 
        initem.write(struct.pack(">i", header_len))  # length
        initem.write(header)                         # record data
        initem.write(struct.pack(">i", header_len))  # length

        # Data arrays 
        tem_level = to_ini[:,:,k].astype('>f8')  
        tem_level = tem_level.T  
        tem_data = tem_level.tobytes()  
        tem_len = len(tem_data)

        initem.write(struct.pack(">i", tem_len))  # length
        initem.write(tem_data)                    # data
        initem.write(struct.pack(">i", tem_len))  # length
